This notebook adapts the 32 prey interaction data to include padding and an updated max velocity from 10 to 25.

# 1.1 Video Dataset

In [ ]:
# mount + imports
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content/drive/MyDrive/Master-Thesis/notebooks/16-32-prey')
sys.path.insert(0, '/content/drive/MyDrive/Master-Thesis/notebooks/16-32-prey')

Mounted at /content/drive


In [ ]:
# pip install
!pip install ultralytics
!pip install deep-sort-realtime

In [ ]:
# import necessary libraries
import os
import cv2
import tqdm
import torch
import pickle
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from utils.dataset_utils import *
from collections import defaultdict
from deep_sort_realtime.deepsort_tracker import DeepSort
from pathlib import Path

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# define paths
yolo_path = Path('/content/drive/MyDrive/Master-Thesis/previous work/models/costumized_yolo/costumized_yolo/costumized_yolo.pt')
raw_video_folder = Path('/content/drive/MyDrive/Predator Prey Thesis/Data/1. Data Processing/Raw/pred_prey_interaction/pred_prey_interaction_32')
processed_video_folder = Path('/content/drive/MyDrive/Predator Prey Thesis/Data/1. Data Processing/Processed/video/expert_tensors/32_prey_interactions')
window_path = Path('/content/drive/MyDrive/Predator Prey Thesis/Data/1. Data Processing/Processed/video/expert_tensors/windows')

total_detections = 33  # number of total detections in frame (pred + prey)
window_len = 10        # length of extracted windows
max_prey = 32

# load device and models
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO(yolo_path)
model.to(device)
tracker = DeepSort(max_age=30, embedder_gpu=True)

pred_tensors_all = []
prey_tensors_all = []

# process each video
for video in os.listdir(raw_video_folder):

    print(f"\nProcessing {video}...")

    video_path = os.path.join(raw_video_folder, video)
    cap = cv2.VideoCapture(video_path)
    total_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # --- total frames ---
    os.makedirs(os.path.join(processed_video_folder, "1. total_frames"), exist_ok=True)
    tf_path = os.path.join(processed_video_folder, "1. total_frames", f"total_frames_{video}.pkl")

    if os.path.exists(tf_path):
        with open(tf_path, "rb") as f:
            total_frames = pickle.load(f)
    else:
        total_frames = []
        for frame in tqdm.tqdm(range(total_frame_count), desc="Processing frames"):
            frame_records = process_frame(cap, model, tracker, frame, device=device)
            total_frames.extend(frame_records)
        cap.release()
        with open(tf_path, "wb") as f:
            pickle.dump(total_frames, f)

    # --- filtered frames ---
    os.makedirs(os.path.join(processed_video_folder, "2. filtered_frames"), exist_ok=True)
    ff_path = os.path.join(processed_video_folder, "2. filtered_frames", f"filtered_frames_{video}.pkl")
    ms_path = os.path.join(processed_video_folder, "2. filtered_frames", f"max_speed_{video}.pkl")

    if os.path.exists(ff_path) and os.path.exists(ms_path):
        with open(ff_path, "rb") as f:
            filtered_frames = pickle.load(f)
        with open(ms_path, "rb") as f:
            max_speed = pickle.load(f)
    else:
        filtered_frames, max_speed = filter_frames(total_frames)
        with open(ff_path, "wb") as f:
            pickle.dump(filtered_frames, f)
        with open(ms_path, "wb") as f:
            pickle.dump(max_speed, f)

    # --- find valid episodes and extract windows ---
    valid_episodes = find_valid_windows(
        filtered_frames, window_len=window_len, total_detections=total_detections
    )

    if not valid_episodes:
        print("No valid episodes found.")
        continue

    extracted_windows = extract_windows(valid_episodes, window_len=window_len)
    print(f"Extracted {len(extracted_windows)} windows with length {window_len}.")

    # --- get expert tensors with new max_speed scaling ---
    ms = 25.0 * width / 2160.0
    pred, prey, coordinates = get_expert_tensors(
        filtered_frames, extracted_windows,
        width, height,
        max_speed=ms, window_size=window_len
    )

    # --- pad up to max_prey and add the active mask channel ---
    pred, prey = pad_expert_tensors(pred, prey, max_prey=max_prey)

    pred_tensors_all.append(pred)
    prey_tensors_all.append(prey)

# concatenate all tensors
pred_tensor = torch.cat(pred_tensors_all, dim=0)
prey_tensor = torch.cat(prey_tensors_all, dim=0)

# add flag feature to prey tensor (marks neighbor slot 0 as the predator)
n, window, agents, neighs, feature = prey_tensor.shape
flag = torch.zeros(
    (n, window, agents, neighs, 1),
    dtype=prey_tensor.dtype,
    device=prey_tensor.device
)
flag[:, :, :, 0, 0] = 1
flag = flag * prey_tensor[..., -2:-1]  # only on active rows
prey_tensor = torch.cat([flag, prey_tensor], dim=-1)

print("\nPredator Tensor:", tuple(pred_tensor.shape))
print("Prey Tensor:", tuple(prey_tensor.shape))

# save
out_folder = window_path / "10 windows (32 prey interactions)"
os.makedirs(out_folder, exist_ok=True)
torch.save(pred_tensor, out_folder / f"pred_tensor_w10_n{len(pred_tensor)}.pt")
torch.save(prey_tensor, out_folder / f"prey_tensor_w10_n{len(prey_tensor)}.pt")

print(f"\nSaved final tensors to: {out_folder}")


Processing pred_prey_interaction_0.07.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.14.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.16.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.15.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.27.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.24.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.17.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.41.mp4...
No valid episodes found.

Processing pred_prey_interaction_0.36.mp4...
No valid episodes found.

Processing pred_prey_interaction_1.11.mp4...
No valid episodes found.

Processing pred_prey_interaction_1.07.mp4...
No valid episodes found.

Processing pred_prey_interaction_1.01.mp4...
No valid episodes found.

Processing pred_prey_interaction_1.09.mp4...
No valid episodes found.

Processing pred_prey_interaction_1.20.mp4...
No valid episodes found.

Proce